In [3]:
from sklearn.model_selection import train_test_split

import pandas as pd

file_path = "../data/raw/PS_20174392719_1491204439457_log.csv"

df = pd.read_csv(file_path)

X = df.drop(columns=["isFraud"])
y = df["isFraud"]

features_to_drop = [
    "nameOrig",
    "nameDest",
    "newbalanceOrig",
    "newbalanceDest",
    "isFlaggedFraud"
]

X = X.drop(columns=features_to_drop)

X = pd.get_dummies(X, columns=["type"], dtype=int)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (6362620, 9)
y shape: (6362620,)


In [5]:
''' Our first feature

Now we create a feature that tells us:
How large is the transaction compared with the sender's available balance?
Formula:
amount / oldbalanceOrg

For example:
Sender balance = ₹100,000
Transaction = ₹10,000 '''

X["amount_to_origin_balance"] = (
    X["amount"] /(X["oldbalanceOrg"] + 1)
)

In [6]:
print(X[[
    "amount",
    "oldbalanceOrg",
    "amount_to_origin_balance"
]].head())

     amount  oldbalanceOrg  amount_to_origin_balance
0   9839.64       170136.0                  0.057834
1   1864.28        21249.0                  0.087731
2    181.00          181.0                  0.994505
3    181.00          181.0                  0.994505
4  11668.14        41554.0                  0.280788


In [7]:
'''Let's create the second feature:
Transaction amount compared with the receiver's balance before the transaction.'''

X["amount_to_destination_balance"] = (
    X["amount"] / (X["oldbalanceDest"] + 1)
)
print(X[[
    "amount",
    "oldbalanceDest",
    "amount_to_destination_balance"
]].head())

'''Your output shows why we used +1:
oldbalanceDest = 0
For example:
9839.64 / (0 + 1) = 9839.64
So there is no division-by-zero error.
What this feature tells us
amount_to_destination_balance represents the transaction amount relative to the receiver's balance before the transaction.
For example:
amount = 181
oldbalanceDest = 21,182
181 / 21,183 ≈ 0.00855
So the transaction is very small compared with the receiver's existing balance.'''

     amount  oldbalanceDest  amount_to_destination_balance
0   9839.64             0.0                    9839.640000
1   1864.28             0.0                    1864.280000
2    181.00             0.0                     181.000000
3    181.00         21182.0                       0.008545
4  11668.14             0.0                   11668.140000


"Your output shows why we used +1:\noldbalanceDest = 0\nFor example:\n9839.64 / (0 + 1) = 9839.64\nSo there is no division-by-zero error.\nWhat this feature tells us\namount_to_destination_balance represents the transaction amount relative to the receiver's balance before the transaction.\nFor example:\namount = 181\noldbalanceDest = 21,182\n181 / 21,183 ≈ 0.00855\nSo the transaction is very small compared with the receiver's existing balance."

In [8]:
'''log transaction amount

Transaction amounts in PaySim have a very wide range. 
A few extremely large transactions can make the distribution difficult for a model to work with.

We can create:
log_amount = log(1 + amount)'''
import numpy as np
X["log_amount"] = np.log1p(X["amount"])

print(X[[
    "amount",
    "log_amount"
]].head())

print(X["log_amount"].describe())

     amount  log_amount
0   9839.64    9.194276
1   1864.28    7.531166
2    181.00    5.204007
3    181.00    5.204007
4  11668.14    9.364703
count    6.362620e+06
mean     1.084087e+01
std      1.814509e+00
min      0.000000e+00
25%      9.502306e+00
50%      1.122355e+01
75%      1.224876e+01
max      1.834213e+01
Name: log_amount, dtype: float64


In [9]:
'''Next: Feature 4 - sender balance remaining
We can create a simple feature representing the sender's balance relative to the transaction:
Send me the output, and we'll check whether this feature makes sense before continuing.'''

X["origin_balance_after_est"] = (
    X["oldbalanceOrg"] - X["amount"]
)
print(X[[
    "amount",
    "oldbalanceOrg",
    "origin_balance_after_est"
]].head())

     amount  oldbalanceOrg  origin_balance_after_est
0   9839.64       170136.0                 160296.36
1   1864.28        21249.0                  19384.72
2    181.00          181.0                      0.00
3    181.00          181.0                      0.00
4  11668.14        41554.0                  29885.86


In [10]:
print(X.columns.tolist())

['step', 'amount', 'oldbalanceOrg', 'oldbalanceDest', 'type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER', 'amount_to_origin_balance', 'amount_to_destination_balance', 'log_amount', 'origin_balance_after_est']


In [ ]:
print(X.shape)

'''Original usable features:
1. step
2. amount
3. oldbalanceOrg
4. oldbalanceDest
5–9. transaction type (one-hot encoded)

Engineered features:
10. amount_to_origin_balance
11. amount_to_destination_balance
12. log_amount
13. origin_balance_after_est'''

(6362620, 13)


In [12]:
''' This will show us how strongly these numerical features are related.'''
print(X[
    [
        "amount",
        "oldbalanceOrg",
        "oldbalanceDest",
        "amount_to_origin_balance",
        "amount_to_destination_balance",
        "log_amount",
        "origin_balance_after_est"
    ]
].corr())

                                 amount  oldbalanceOrg  oldbalanceDest  \
amount                         1.000000      -0.002762        0.294137   
oldbalanceOrg                 -0.002762       1.000000        0.066243   
oldbalanceDest                 0.294137       0.066243        1.000000   
amount_to_origin_balance       0.817079      -0.040134        0.307827   
amount_to_destination_balance  0.273904      -0.022608       -0.048045   
log_amount                     0.387260       0.106980        0.227789   
origin_balance_after_est      -0.207239       0.978859        0.004643   

                               amount_to_origin_balance  \
amount                                         0.817079   
oldbalanceOrg                                 -0.040134   
oldbalanceDest                                 0.307827   
amount_to_origin_balance                       1.000000   
amount_to_destination_balance                 -0.007689   
log_amount                                     0.2125

In [ ]:
'''The main thing I notice is:
oldbalanceOrg ↔ origin_balance_after_est = 0.978859
That's very high because:
origin_balance_after_est = oldbalanceOrg - amount
So these two features contain almost the same information.
What should we do?
For our project, let's keep the feature set simple and interpretable.
I recommend removing origin_balance_after_est for now.'''
X = X.drop(columns=["origin_balance_after_est"])
print(X.columns.tolist())
print("Shape:", X.shape)

'''step
amount
oldbalanceOrg
oldbalanceDest

type_CASH_IN
type_CASH_OUT
type_DEBIT
type_PAYMENT
type_TRANSFER

amount_to_origin_balance
amount_to_destination_balance
log_amount'''

['step', 'amount', 'oldbalanceOrg', 'oldbalanceDest', 'type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER', 'amount_to_origin_balance', 'amount_to_destination_balance', 'log_amount']
Shape: (6362620, 12)


In [14]:
#final check for notebook 4
print("Total missing values:", X.isnull().sum().sum())
print(X.dtypes)
print("Final X shape:", X.shape)
print("Target y shape:", y.shape)

Total missing values: 0
step                               int64
amount                           float64
oldbalanceOrg                    float64
oldbalanceDest                   float64
type_CASH_IN                       int64
type_CASH_OUT                      int64
type_DEBIT                         int64
type_PAYMENT                       int64
type_TRANSFER                      int64
amount_to_origin_balance         float64
amount_to_destination_balance    float64
log_amount                       float64
dtype: object
Final X shape: (6362620, 12)
Target y shape: (6362620,)


In [ ]:
'''Our Notebook 04 X is the full dataset. We haven't recreated X_train/X_test from these engineered features.

So do not start model training yet.
We need to make sure the train/test split contains these final 12 features. 
Since we previously split the 9-feature version in Notebook 03, those old X_train and X_test don't contain our new features.

Next
We'll move to Notebook 05 — Model Training, but first we'll create the correct train/test split using these final engineered features.
And because fraud is extremely imbalanced (~1 fraud per 774 normal transactions), we'll handle that carefully when training.
Notebook 04 is done. 🎯'''